# Cambodia Digital Economy & Cyber Readiness: Exploratory Data Analysis

**Question:** how have Cambodia's economy and digital adoption developed since 2000,
and how does Cambodia compare with Vietnam, Myanmar and Laos?

**Data:** World Bank, World Development Indicators, collected by `fetch_data.py`
(retrieval time in `data/metadata.json`). Missing observations are kept as missing
(NaN); nothing is estimated or filled in.

**Sections:** A. Data inspection · B. Descriptive statistics · C. Economic growth ·
D. Digital development · E. Correlation · F. Findings

> Charts are interactive Plotly figures. They render in Jupyter, VS Code and nbviewer,
> but not in GitHub's notebook preview.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

DATA_DIR = Path("..") / "data"
df = pd.read_csv(DATA_DIR / "cambodia_econ.csv")
metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))

COUNTRY_ORDER = ["Cambodia", "Vietnam", "Myanmar", "Laos"]
# One fixed colour per country, used in every chart (colour-blind-checked order).
COUNTRY_COLORS = {"Cambodia": "#2a78d6", "Vietnam": "#eb6834", "Myanmar": "#1baf7a", "Laos": "#eda100"}

print("Data retrieved (UTC):", metadata["retrieved_at_utc"])
print("World Bank database last updated:", metadata["indicators"]["NY.GDP.PCAP.CD"]["database_last_updated"])

Data retrieved (UTC): 2026-11-19T15:01:53+00:00
World Bank database last updated: 2026-11-11


In [2]:
def style_chart(fig, title, y_label):
    """Apply one consistent, quiet style to every chart."""
    fig.update_layout(
        title=title, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
        font=dict(family="system-ui, Segoe UI, sans-serif", color="#0b0b0b", size=13),
        legend_title_text="", hovermode="x unified", margin=dict(t=60, r=20, b=40, l=60),
    )
    fig.update_xaxes(title="Year", showgrid=False, linecolor="#c3c2b7", ticks="outside", tickcolor="#c3c2b7")
    fig.update_yaxes(title=y_label, gridcolor="#e1e0d9", zeroline=False, linecolor="#c3c2b7")
    return fig


def line_chart(indicator_code, log_y=False):
    """Line chart of one indicator for all four countries. Missing years show as gaps."""
    info = metadata["indicators"][indicator_code]
    data = df[df["indicator_code"] == indicator_code]
    fig = px.line(
        data, x="year", y="value", color="country", log_y=log_y,
        category_orders={"country": COUNTRY_ORDER}, color_discrete_map=COUNTRY_COLORS,
    )
    fig.update_traces(line_width=2, hovertemplate="%{y:,.2f}")
    if log_y:
        fig.update_yaxes(dtick=1)   # label only powers of ten: 0.1, 1, 10, 100 ...
    return style_chart(fig, f"{info['name']} ({info['unit']})", info["unit"])


def wide_table(indicator_code):
    """One indicator as a table: rows = years, columns = countries."""
    data = df[df["indicator_code"] == indicator_code]
    return data.pivot(index="year", columns="country", values="value")[COUNTRY_ORDER]

## A. Data inspection

Before analysing anything, confirm the dataset is what we expect: its shape,
column types, missing values and duplicates.

In [3]:
df.head()

,country,country_code,year,indicator,indicator_code,value
0,Cambodia,KHM,2000,FDI net inflows,BX.KLT.DINV.WD.GD.ZS,3.202576
1,Cambodia,KHM,2001,FDI net inflows,BX.KLT.DINV.WD.GD.ZS,3.533377
2,Cambodia,KHM,2002,FDI net inflows,BX.KLT.DINV.WD.GD.ZS,2.909348
3,Cambodia,KHM,2003,FDI net inflows,BX.KLT.DINV.WD.GD.ZS,1.616517
4,Cambodia,KHM,2004,FDI net inflows,BX.KLT.DINV.WD.GD.ZS,2.233717


In [4]:
print("Rows, columns:", df.shape)
print("Countries:", df["country"].unique().tolist())
print("Indicators:", df["indicator"].nunique())
print("Years:", df["year"].min(), "to", df["year"].max())
df.dtypes

Rows, columns: (832, 6)
Countries: ['Cambodia', 'Laos', 'Myanmar', 'Vietnam']
Indicators: 8
Years: 2000 to 2025


country               str
country_code          str
year                int64
indicator             str
indicator_code        str
value             float64
dtype: object

In [5]:
# Missing values per indicator. Missing means "not published", never zero.
missing = df.groupby("indicator")["value"].agg(rows="size", missing=lambda s: s.isna().sum())
missing["missing_pct"] = (missing["missing"] / missing["rows"] * 100).round(1)
missing.sort_values("missing", ascending=False)

,rows,missing,missing_pct
indicator,,,
Secure Internet servers,104,44,42.3
Remittances received,104,27,26.0
Internet users,104,9,8.7
Inflation,104,6,5.8
Mobile subscriptions,104,5,4.8
GDP per capita,104,0,0.0
FDI net inflows,104,0,0.0
GDP growth,104,0,0.0


In [6]:
duplicates = df.duplicated(subset=["country_code", "year", "indicator_code"]).sum()
print("Duplicate country-year-indicator rows:", duplicates)

Duplicate country-year-indicator rows: 0


In [7]:
# Latest year with a real value. Indicators and countries differ.
latest_year = df.dropna(subset=["value"]).groupby(["indicator", "country"])["year"].max().unstack()
latest_year[COUNTRY_ORDER]

country,Cambodia,Vietnam,Myanmar,Laos
indicator,,,,
FDI net inflows,2025,2025,2025,2025
GDP growth,2025,2025,2025,2025
GDP per capita,2025,2025,2025,2025
Inflation,2025,2025,2019,2025
Internet users,2024,2024,2020,2024
Mobile subscriptions,2024,2024,2024,2023
Remittances received,2025,2004,2019,2025
Secure Internet servers,2024,2024,2024,2024


## B. Descriptive statistics

`count` is the number of real observations. Always read it first: a mean based on
5 years is not comparable with a mean based on 26 years.

In [8]:
stats = df.groupby(["indicator", "country"])["value"].agg(["count", "mean", "median", "min", "max", "std"])
stats.round(2)

count     mean   median     min      max  \
indicator               country                                              
FDI net inflows         Cambodia     26     7.72     9.14    1.62    11.15   
                        Laos         26     5.06     5.44    0.25    11.24   
                        Myanmar      26     2.99     2.51    1.31     7.27   
                        Vietnam      26     4.79     4.28    3.39     9.66   
GDP growth              Cambodia     26     7.17     7.56   -3.56    13.30   
                        Laos         26     6.15     6.63    0.50     8.62   
                        Myanmar      26     6.77     7.75  -12.02    13.84   
                        Vietnam      26     6.39     6.74    2.55     8.54   
GDP per capita          Cambodia     26  1353.99  1249.23  296.43  2872.43   
                        Laos         26  1491.85  1688.04  313.35  2589.32   
                        Myanmar      26   908.08  1182.91  141.53  1490.22   
                        Vietnam      26  2242.83  2272.32  404.03  5065.99   
Inflation               Cambodia     26     3.67     2.93   -1.24    24.10   
                        Laos         26     8.73     6.46    0.14    31.23   
                        Myanmar      20    13.67     7.32   -0.11    57.07   
                        Vietnam      26     5.61     3.58   -1.71    23.12   
Internet users          Cambodia     25    21.66     4.94    0.05    68.93   
                        Laos         25    21.10    10.75    0.11    65.59   
                        Myanmar      20     8.73     0.62    0.00    45.45   
                        Vietnam      25    39.12    36.80    0.25    84.15   
Mobile subscriptions    Cambodia     25    75.63   113.39    1.05   133.58   
                        Laos         24    43.08    53.84    0.23    85.28   
                        Myanmar      25    49.90     7.48    0.03   155.67   
                        Vietnam      25    94.15   127.61    1.02   148.50   
Remittances received    Cambodia     26     4.16     4.30    1.14     7.43   
                        Laos         26     0.98     1.21    0.03     2.58   
                        Myanmar      20     1.66     1.13    0.15     3.94   
                        Vietnam       5     4.62     5.05    3.37     5.31   
Secure Internet servers Cambodia     15   127.86    54.94    0.69   817.29   
                        Laos         15    71.54    16.24    0.47   301.68   
                        Myanmar      15     9.41    10.19    0.02    23.60   
                        Vietnam      15  1911.16  1340.49    2.31  5733.75   

                                      std  
indicator               country            
FDI net inflows         Cambodia     2.87  
                        Laos         2.86  
                        Myanmar      1.47  
                        Vietnam      1.49  
GDP growth              Cambodia     3.17  
                        Laos         2.05  
                        Myanmar      6.73  
                        Vietnam      1.37  
GDP per capita          Cambodia   802.72  
                        Laos       861.15  
                        Myanmar    491.28  
                        Vietnam   1466.32  
Inflation               Cambodia     4.78  
                        Laos         8.18  
                        Myanmar     14.85  
                        Vietnam      5.36  
Internet users          Cambodia    27.59  
                        Laos        24.06  
                        Myanmar     14.02  
                        Vietnam     27.60  
Mobile subscriptions    Cambodia    55.09  
                        Laos        27.73  
                        Myanmar     58.19  
                        Vietnam     58.95  
Remittances received    Cambodia     1.85  
                        Laos         0.78  
                        Myanmar      1.36  
                        Vietnam      0.80  
Secure Internet servers Cambodia   207.87  
                 

In [9]:
# Latest available value for each indicator and country, with its year.
latest = df.dropna(subset=["value"]).sort_values("year").groupby(["indicator", "country"]).tail(1)
latest["latest"] = latest["value"].map("{:,.1f}".format) + " (" + latest["year"].astype(str) + ")"
latest.pivot(index="indicator", columns="country", values="latest")[COUNTRY_ORDER]

country,Cambodia,Vietnam,Myanmar,Laos
indicator,,,,
FDI net inflows,9.9 (2025),4.3 (2025),1.3 (2025),7.7 (2025)
GDP growth,5.3 (2025),8.0 (2025),-2.0 (2025),4.5 (2025)
GDP per capita,"2,872.4 (2025)","5,066.0 (2025)","1,488.9 (2025)","2,324.8 (2025)"
Inflation,2.4 (2025),3.3 (2025),8.8 (2019),7.7 (2025)
Internet users,68.5 (2024),84.2 (2024),45.4 (2020),65.6 (2024)
Mobile subscriptions,116.1 (2024),127.6 (2024),114.3 (2024),64.8 (2023)
Remittances received,4.3 (2025),5.1 (2004),3.4 (2019),2.6 (2025)
Secure Internet servers,817.3 (2024),"5,733.7 (2024)",23.6 (2024),301.7 (2024)


## C. Economic growth

GDP growth is the annual % change of real GDP (constant prices), so it is not
distorted by inflation. Values below zero are contractions.

In [10]:
fig = line_chart("NY.GDP.MKTP.KD.ZG")
fig.add_hline(y=0, line_width=1, line_color="#898781")
fig.show()

In [11]:
growth = df[df["indicator_code"] == "NY.GDP.MKTP.KD.ZG"].dropna(subset=["value"])


def describe_growth(group):
    best = group.loc[group["value"].idxmax()]
    worst = group.loc[group["value"].idxmin()]
    contractions = group.loc[group["value"] < 0, "year"].astype(str)
    return pd.Series({
        "mean_growth_pct": round(group["value"].mean(), 2),
        "best_year": int(best["year"]), "best_pct": round(best["value"], 2),
        "worst_year": int(worst["year"]), "worst_pct": round(worst["value"], 2),
        "contraction_years": ", ".join(contractions) or "none",
    })


growth.groupby("country").apply(describe_growth).loc[COUNTRY_ORDER]

,mean_growth_pct,best_year,best_pct,worst_year,worst_pct,contraction_years
country,,,,,,
Cambodia,7.17,2005,13.30,2020,-3.56,2020
Vietnam,6.39,2022,8.54,2021,2.55,none
Myanmar,6.77,2003,13.84,2021,-12.02,"2020, 2021, 2024, 2025"
Laos,6.15,2006,8.62,2020,0.50,none


In [12]:
# Average growth by period (decades are neutral boundaries, chosen before looking at results).
growth["period"] = pd.cut(growth["year"], bins=[1999, 2009, 2019, 2100],
                          labels=["2000-2009", "2010-2019", "2020-latest"])
growth.pivot_table(index="country", columns="period", values="value", aggfunc="mean", observed=True).loc[COUNTRY_ORDER].round(2)

period,2000-2009,2010-2019,2020-latest
country,,,
Cambodia,8.96,7.58,3.51
Vietnam,6.65,6.58,5.67
Myanmar,12.40,7.10,-3.18
Laos,6.85,7.31,3.03


### GDP per capita

GDP per capita in **current US$** is not adjusted for inflation or exchange-rate
changes, so it is good for comparing countries in the same year but overstates
real growth over time.

In [13]:
line_chart("NY.GDP.PCAP.CD").show()

In [14]:
gdp_pc = wide_table("NY.GDP.PCAP.CD")
latest_gdp_year = gdp_pc.index.max()
snapshot = gdp_pc.loc[[2000, 2010, 2020, latest_gdp_year]].round(0)
snapshot.loc["ratio to Cambodia (latest)"] = (gdp_pc.loc[latest_gdp_year] / gdp_pc.loc[latest_gdp_year, "Cambodia"]).round(2)
snapshot

country,Cambodia,Vietnam,Myanmar,Laos
year,,,,
2000,296.0,404.00,197.00,319.00
2010,952.0,1683.00,1011.00,1126.00
2020,2082.0,3534.00,1490.00,2584.00
2025,2872.0,5066.00,1489.00,2325.00
ratio to Cambodia (latest),1.0,1.76,0.52,0.81


## D. Digital development

**Internet users**: share of the population that used the Internet in the last
3 months (ITU). **Mobile subscriptions**: subscriptions per 100 people; can exceed
100 because one person may own several SIM cards.

In [15]:
line_chart("IT.NET.USER.ZS").show()

In [16]:
def milestone_table(indicator_code, threshold, label):
    """Values in 2000 and 2010, the latest real value, and the first year a threshold was reached."""
    table = wide_table(indicator_code)
    return pd.DataFrame({
        "2000": table.loc[2000],
        "2010": table.loc[2010],
        "latest_value": table.apply(lambda s: s.dropna().iloc[-1]),
        "latest_year": table.apply(lambda s: s.last_valid_index()),
        label: table.apply(lambda s: s.index[s >= threshold].min()).astype("Int64"),  # <NA> = never reached
    }).round(2)


milestone_table("IT.NET.USER.ZS", 50, "first_year_at_or_above_50pct")

,2000,2010,latest_value,latest_year,first_year_at_or_above_50pct
country,,,,,
Cambodia,0.05,1.26,68.48,2024,2018
Vietnam,0.25,30.65,84.15,2024,2016
Myanmar,NaN,0.25,45.45,2020,<NA>
Laos,0.11,7.00,65.59,2024,2020


In [17]:
line_chart("IT.CEL.SETS.P2").show()

In [18]:
milestone_table("IT.CEL.SETS.P2", 100, "first_year_at_or_above_100")

,2000,2010,latest_value,latest_year,first_year_at_or_above_100
country,,,,,
Cambodia,1.05,56.21,116.09,2024,2012
Vietnam,1.02,127.57,127.61,2024,2009
Myanmar,0.03,1.21,114.33,2024,2018
Laos,0.23,63.20,64.76,2023,<NA>


### Secure Internet servers

Definition (World Bank / Netcraft): distinct, publicly-trusted TLS/SSL certificates
found by Netcraft's Secure Server Survey, **by hosting country**, per 1 million people.
It is a **digital infrastructure** indicator. It is **not** a measure of
cybersecurity readiness or national cyber resilience. Data start in 2010.

Values span four orders of magnitude, so the chart uses a **logarithmic** y-axis:
equal vertical distances mean equal multiplication (x10), not equal addition.

In [19]:
line_chart("IT.NET.SECR.P6", log_y=True).show()

In [20]:
servers = wide_table("IT.NET.SECR.P6")
servers.loc[[2010, 2015, 2020, servers.last_valid_index()]].round(2)

country,Cambodia,Vietnam,Myanmar,Laos
year,,,,
2010,0.69,2.31,0.02,0.47
2015,10.11,32.68,1.08,4.12
2020,188.46,3082.34,14.43,52.27
2024,817.29,5733.75,23.60,301.68


In [21]:
# Largest single-year change per country: big jumps can come from how the survey counts certificates.
yearly_change_pct = servers.pct_change(fill_method=None) * 100   # fill_method=None: never fill gaps
pd.DataFrame({"year_of_largest_rise": yearly_change_pct.idxmax(), "rise_pct": yearly_change_pct.max().round(0)})

,year_of_largest_rise,rise_pct
country,,
Cambodia,2024,266.0
Vietnam,2016,750.0
Myanmar,2016,1861.0
Laos,2022,233.0


## E. Correlation: Internet users vs GDP per capita

**Pearson's r** measures how closely two variables follow a straight-line
relationship, from -1 (perfect negative) through 0 (no linear relationship) to +1
(perfect positive).

Rules for reading it:
- **Correlation is not causation.** Both variables could be driven by a third factor,
  or the causality could run either way.
- **Shared time trends inflate r.** Internet use and GDP per capita both rise over
  time in all four countries, so a strong r partly reflects "both went up since 2000".
- **Only complete pairs count.** A country-year enters only if both values exist,
  so the sample size `n` must be reported. Observations of the same country in
  consecutive years are not independent, so this is description, not a formal test.

In [22]:
pairs = (
    df[df["indicator_code"].isin(["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"])]
    .pivot(index=["country", "year"], columns="indicator_code", values="value")
    .rename(columns={"IT.NET.USER.ZS": "internet_users_pct", "NY.GDP.PCAP.CD": "gdp_per_capita_usd"})
    .dropna()            # keep only country-years where BOTH values exist
    .reset_index()
)
print("Complete country-year pairs (n):", len(pairs))
pairs.groupby("country").size().loc[COUNTRY_ORDER]

Complete country-year pairs (n): 95


country
Cambodia    25
Vietnam     25
Myanmar     20
Laos        25
dtype: int64

In [23]:
fig = px.scatter(
    pairs, x="internet_users_pct", y="gdp_per_capita_usd", color="country", facet_col="country",
    category_orders={"country": COUNTRY_ORDER}, color_discrete_map=COUNTRY_COLORS,
    hover_data={"year": True, "country": False},
    labels={"internet_users_pct": "Internet users (% of population)",
            "gdp_per_capita_usd": "GDP per capita (current US$)"},
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="#fcfcfb")))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))   # "country=Laos" -> "Laos"
style_chart(fig, "Internet users vs GDP per capita, one point per country-year", "GDP per capita (current US$)")
fig.update_xaxes(title="Internet users (%)")
fig.update_yaxes(title=None)                                        # shared axes: title only once,
fig.update_yaxes(title="GDP per capita (current US$)", col=1)       # on the first panel
fig.update_layout(hovermode="closest", showlegend=False)
fig.show()

In [24]:
internet_pct, gdp_pc_usd = pairs["internet_users_pct"], pairs["gdp_per_capita_usd"]
pooled_pearson = internet_pct.corr(gdp_pc_usd)                   # Pearson is the default method
pooled_spearman = internet_pct.rank().corr(gdp_pc_usd.rank())    # Spearman = Pearson on ranks
print(f"All four countries pooled: Pearson r = {pooled_pearson:.2f}, Spearman rho = {pooled_spearman:.2f}, n = {len(pairs)}")

by_country = pairs.groupby("country").apply(
    lambda g: pd.Series({"pearson_r": g["internet_users_pct"].corr(g["gdp_per_capita_usd"]), "n": len(g)})
)
by_country.loc[COUNTRY_ORDER].astype({"n": int}).round(2)

All four countries pooled: Pearson r = 0.90, Spearman rho = 0.93, n = 95


,pearson_r,n
country,,
Cambodia,0.92,25
Vietnam,0.99,25
Myanmar,0.67,20
Laos,0.80,25


*Spearman's rho* ranks the values first, so it captures any steadily increasing
relationship, not only a straight line, and is less sensitive to extreme values.

## F. Findings

All figures come from the outputs above (World Bank WDI, retrieved 2026-11-19).
They describe what the data show; they do not explain why.

1. **Cambodia had the highest average growth of the four, and slower growth since 2020.**
   Real GDP growth averaged 7.17% a year over 2000-2025 (Myanmar 6.77%, Vietnam 6.39%,
   Laos 6.15%). Cambodia's period average fell from 8.96% (2000-2009) to 7.58% (2010-2019)
   and 3.51% (2020-2025). Its only contraction was 2020 (-3.56%), followed by 5.1%, 5.0%,
   6.0% and 5.3% growth in 2022-2025. Myanmar contracted in 2020, 2021 (-12.02%), 2024 and
   2025; Vietnam and Laos had no contraction year.

2. **Cambodia's GDP per capita is second of the four and has been above Laos since 2022.**
   In current US$ it rose from \$296 (2000) to \$2,872 (2025), 9.7 times higher (not
   adjusted for inflation). Laos was higher in 2000, 2010 and 2021; Cambodia has been higher
   every year since 2022 (\$2,325 vs \$2,046 in 2022). In 2025 Vietnam's GDP per capita was
   1.76 times Cambodia's, Laos 0.81 times and Myanmar 0.52 times.

3. **Internet use in Cambodia went from almost zero to about two-thirds of the population.**
   Internet users rose from 0.05% (2000) to 1.26% (2010) and 68.48% (2024). Cambodia passed
   50% in 2018, two years after Vietnam (2016) and two years before Laos (2020); Vietnam is
   highest at 84.15% (2024). Myanmar's latest value is 45.45% in 2020, with no data after
   that, so it cannot be compared for recent years. Mobile subscriptions in Cambodia passed
   100 per 100 people in 2012 (Vietnam 2009, Myanmar 2018); Laos had not reached 100 by its
   latest year (64.76 in 2023).

4. **Secure Internet servers grew from a very low base, and Vietnam is far ahead.**
   Cambodia rose from 0.69 per million people (2010) to 817.29 (2024), including a jump
   from 223.2 in 2023 to 817.3 in 2024 (+266%). Vietnam's 2024 value (5,733.7) is 7.0 times
   Cambodia's; Laos is at 301.7 and Myanmar at 23.6. Single-year jumps this large (Myanmar
   +1,861% in 2016) call for caution. The indicator counts TLS/SSL certificates **by hosting
   country**, so it reflects where secure websites are hosted, not how well a country is
   protected against cyber threats.

5. **Internet use and GDP per capita are strongly and positively associated.**
   Across 95 country-years with both values, Pearson r = 0.90 (Spearman rho = 0.93).
   Within each country: Vietnam 0.99, Cambodia 0.92, Laos 0.80, Myanmar 0.67 (n = 20).
   Both variables rise over time in all four countries, so the association partly reflects
   shared upward trends. It does not show that Internet adoption raises income, or the reverse.

### Limitations

- The most recent values (2024-2025) may be revised by the World Bank in later releases.
- GDP per capita in current US\$ is affected by inflation and exchange rates.
- Gaps: Vietnam remittances after 2004, Myanmar inflation after 2019 and Internet users
  after 2020, Secure Internet servers before 2010.
- Four countries over about 25 years is a small sample, and consecutive years of the same
  country are not independent observations, so the correlations are descriptive only.